In [8]:
def dfa(input: str):
    flag = True
    for char in input:
        
        if char.islower():
            continue
        else:
            flag = False
            break
    
    if flag:
        print("Accepted")
        
    else :
        print("Not Accepted")
        

In [9]:
word = "1apple"
dfa(word)

Not Accepted


In [10]:
from collections import defaultdict


# ============================================================
# 1. MORPHOLOGICAL CLASSES
# ============================================================

VOWELS = set("aeiou")


def get_ending_class(word):
    """
    Determine which pluralization rule applies to a noun.
    """

    # consonant + y
    if len(word) >= 2 and word.endswith("y"):
        if word[-2].lower() not in VOWELS:
            return "Y"

        # vowel + y
        return "VY"

    # words ending in s, z, x, ch, sh
    if (
        word.endswith("s")
        or word.endswith("z")
        or word.endswith("x")
        or word.endswith("ch")
        or word.endswith("sh")
    ):
        return "SZXC"

    # normal noun
    return "NORMAL"


# ============================================================
# 2. TRIE NODE
# ============================================================

class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False
        self.word_class = None


# 3. FST

class NounFST:

    def __init__(self, noun_file):
        self.root = TrieNode()

        self.load_nouns(noun_file)

    # --------------------------------------------------------
    # Load nouns into trie
    # --------------------------------------------------------

    def load_nouns(self, noun_file):

        with open(noun_file, "r", encoding="utf-8") as f:

            for line in f:

                word = line.strip().lower()

                if not word:
                    continue

                # Some datasets may contain extra spaces
                word = word.split()[0]

                self.insert(word)

    # --------------------------------------------------------
    # Insert noun into trie
    # --------------------------------------------------------

    def insert(self, word):

        node = self.root

        for ch in word:

            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]

        node.is_word = True
        node.word_class = get_ending_class(word)

    # --------------------------------------------------------
    # Find exact noun in trie
    # --------------------------------------------------------

    def find_word(self, word):

        node = self.root

        for ch in word:

            if ch not in node.children:
                return None

            node = node.children[ch]

        if node.is_word:
            return node

        return None

    # ========================================================
    # 4. ANALYZE INPUT WORD
    # ========================================================

    def analyze(self, word):

        word = word.lower().strip()

        # ----------------------------------------------------
        # CASE 1: Word itself is a noun
        # ----------------------------------------------------

        node = self.find_word(word)

        if node is not None:

            return word + "+N+SG"

        # ----------------------------------------------------
        # CASE 2: Regular plural
        #
        # Example:
        # bag -> bags
        # boy -> boys
        # ----------------------------------------------------

        if word.endswith("s"):

            stem = word[:-1]

            node = self.find_word(stem)

            if node is not None:

                # Regular noun
                if node.word_class == "NORMAL":

                    return stem + "+N+PL"

                # vowel + y
                if node.word_class == "VY":

                    return stem + "+N+PL"

        # ----------------------------------------------------
        # CASE 3: E insertion
        #
        # fox -> foxes
        # watch -> watches
        # dish -> dishes
        # buzz -> buzzes
        # ----------------------------------------------------

        if word.endswith("es"):

            stem = word[:-2]

            node = self.find_word(stem)

            if node is not None:

                if node.word_class == "SZXC":

                    return stem + "+N+PL"

        # ----------------------------------------------------
        # CASE 4: Y replacement
        #
        # try -> tries
        # cry -> cries
        # fly -> flies
        # ----------------------------------------------------

        if word.endswith("ies"):

            stem = word[:-3] + "y"

            node = self.find_word(stem)

            if node is not None:

                if node.word_class == "Y":

                    return stem + "+N+PL"

        # ----------------------------------------------------
        # INVALID
        # ----------------------------------------------------

        return "Invalid Word"


# ============================================================
# 5. TEST
# ============================================================

if __name__ == "__main__":

    fst = NounFST("brown_nouns.txt")

    test_words = [
        "fox",
        "foxes",
        "foxs",
        "bag",
        "bags",
        "watch",
        "watches",
        "try",
        "tries",
        "boy",
        "boys",
        "dish",
        "dishes",
        "dishs"
    ]

    for word in test_words:

        result = fst.analyze(word)

        print(f"{word} = {result}")

fox = fox+N+SG
foxes = fox+N+PL
foxs = Invalid Word
bag = bag+N+SG
bags = bags+N+SG
watch = watch+N+SG
watches = watch+N+PL
try = try+N+SG
tries = tries+N+SG
boy = boy+N+SG
boys = boys+N+SG
dish = dish+N+SG
dishes = dishes+N+SG
dishs = Invalid Word
